In [1]:
%matplotlib inline
import internal_script # Allow absolute imports in internal scripts

from dorion_francois.numerical_methods import *

## Longstaff Schwartz

There are two approaches to implementing the Longstaff and Schwartz (LS) algorithm: one closer to the example in
the original LS paper, and one closer to the slides I presented before the paper's example. 

Below, I'll use an example from Dorion & Francois (2023-02, Chapter 6, Problem 6) to illustrate both methods. I'll start with the original LS approach, and then turn to what I believe is a simpler approach. Both yield the same result (phew) and are equally valid in, say, the final exam :)

### The Problem

The Monte Carlo simulation generated the five following paths:

In [2]:
paths = problem06_paths()
show_hide_cell()

,month 0,month 1,month 2,month 3,month 4
trajectory 1,100.0,101.27,99.05,99.97,102.14
trajectory 2,100.0,98.82,100.57,103.11,102.90
trajectory 3,100.0,100.05,101.44,98.04,96.56
trajectory 4,100.0,102.22,103.29,101.55,104.05
trajectory 5,100.0,98.53,99.36,100.98,97.64


The goal is to evaluate the ATM, 4-month American call written on the
simulated UA using the Longstaff and Schwartz (2001) regression approach. The risk-free rate is 3\%.

Suppose the regressions estimating the continuation values are the following

$$
\begin{matrix}
0.002S^{2}-0.12S-7 &  & \textrm{for $t=3$ months,} \\ 
0.001S^{2}-0.08S-1 &  & \textrm{for $t=2$ months,} \\ 
0.001S^{2}-0.12S+3 &  & \textrm{for $t=1$ month.}
\end{matrix}%
$$

a) For each path, determine the optimal exercise date.

b) Compute the initial value of the call.

In [3]:
disc = np.exp(-0.03 * 1/12) # discount over one period

# S0 = 100, ATM call
K = 100 

# Although the paths are visually represented as rows in the book, 
# coding convention herein dictates that time increase with rows 
# (as numpy defaults to row-major ordering), and columns be paths.
stock = paths.to_numpy().T
exercise = np.maximum(0, stock-K)

First, we populate the last column with the terminal exercise value.

In [4]:
payoff = np.full(stock.shape, np.nan)
payoff[4,:] = exercise[4,:]

# The print_paths function transposes paths back to visualization format 
# by default (can be overriden)
print_paths(payoff, hide=np.nan);

,t=0dt,t=1dt,t=2dt,t=3dt,t=4dt
trajectory 1,,,,,2.14
trajectory 2,,,,,2.90
trajectory 3,,,,,0.00
trajectory 4,,,,,4.05
trajectory 5,,,,,0.00


### At month 3
This should be in a for loop, but I'll unfold it here for pedagogical purposes

To obtain the continuation value on paths that are in the money (ITM) at t=3, we apply the polynomial
at the current values of the underlying

In [5]:
ITM = np.where(exercise[3,:] > 0)[0] # indices of the ITM trajectories
print('ITM paths:',1+ITM) # Do not forget that Python indices start at 0...

ITM paths: [2 4 5]


In [6]:
S = stock[3,ITM]
X = np.array([S**2, S, np.ones(S.shape)])
coeffs = np.array([0.002, -0.12, -7]).reshape(-1,1)
continuation = X.T.dot(coeffs).T[0,:]
continuation

array([1.8901442, 1.438805 , 1.2763208])

Then, on trajectories that are **both** ITM and such that early exercise is preferable to the *approximated* continuation value, "exercise" the option, and get rid of any later exercises, if applicable.

In [7]:
early = exercise[3,ITM] > continuation
payoff[3,ITM[early]] = exercise[3,ITM[early]]
payoff[4:,ITM[early]] = np.nan # Erase later exercises if any (cf. the nansum below)
print_paths(payoff, hide=np.nan);

,t=0dt,t=1dt,t=2dt,t=3dt,t=4dt
trajectory 1,,,,,2.14
trajectory 2,,,,3.11,
trajectory 3,,,,,0.0
trajectory 4,,,,1.55,
trajectory 5,,,,,0.0


### At month 2

In [8]:
ITM = np.where(exercise[2,:] > 0)[0] # indices of the ITM trajectories
print('ITM paths:',1+ITM) # Do not forget that Python indices start at 0...

ITM paths: [2 3 4]


To better understand the algorithm, although the coefficients are given in the question above,
we'll procede as if they were not for now. The states ($X$) are easy to get:

In [9]:
S = stock[3,ITM]
X = np.array([S**2, S, np.ones(S.shape)])

But the expected payoff are a tad more work

In [10]:
Y = payoff[3:,ITM] # The payoffs in the future, not all at same time
# disc, one time step in the future, vs disc**2 two time steps in the future
D = np.array([disc, disc**2]).reshape(-1,1) 
y = np.nansum((Y*D),axis=0) # This "sum" returns the only non-nan entry
X,y

(array([[1.06316721e+04, 9.61184160e+03, 1.03124025e+04],
        [1.03110000e+02, 9.80400000e+01, 1.01550000e+02],
        [1.00000000e+00, 1.00000000e+00, 1.00000000e+00]]),
 array([3.10223471, 0.        , 1.54612984]))

Now this example is a bit unfortunate because the payoff in trajectory 3 is 0, 
so if we erronously discounted with `disc` instead of `disc**2`, it would yield a correct answer... merely out of luck!

In [11]:
if False:
    # Conceptually, we'd find the coefficients as follows
    import statsmodels.api as sm
    model = sm.OLS(y, X.T).fit()
    coeffs = model.params
else:
    # but there are too few trajectories so the results could make little 
    # sense. Let's stick with the coefficients provided in the question
    coeffs = np.array([0.001, -0.08, -1]).reshape(-1,1)

In [12]:
continuation = X.T.dot(coeffs).T[0,:]
early = exercise[2,ITM] > continuation
payoff[2,ITM[early]] = exercise[2,ITM[early]]
payoff[3:,ITM[early]] = np.nan # Erase later exercises if any
print_paths(payoff, hide=np.nan);

,t=0dt,t=1dt,t=2dt,t=3dt,t=4dt
trajectory 1,,,,,2.14
trajectory 2,,,,3.11,
trajectory 3,,,1.44,,
trajectory 4,,,3.29,,
trajectory 5,,,,,0.0


### At month 1

In [13]:
ITM = np.where(exercise[1,:] > 0)[0] # indices of the ITM trajectories
print('ITM paths:',1+ITM) # Do not forget that Python indices start at 0...

ITM paths: [1 3 4]


In [14]:
# The coefficients are given in the question above; again, for illustration purposes:
S = stock[3,ITM]
X = np.array([S**2, S, np.ones(S.shape)])

Y = payoff[2:,ITM] # The payoffs in the future, not all at same time
D = np.vstack((D, D[-1,:]*disc))
y = np.nansum((Y*D),axis=0)
X,y

(array([[9.99400090e+03, 9.61184160e+03, 1.03124025e+04],
        [9.99700000e+01, 9.80400000e+01, 1.01550000e+02],
        [1.00000000e+00, 1.00000000e+00, 1.00000000e+00]]),
 array([2.12401004, 1.4364045 , 3.28178527]))

Here the payoff of trajectory 1 is non-null; discounting with 
anything else than `disc**3` (now the third row of D) would yield wrong y's.
Again, let's go back to the coefficients in the question.

In [15]:
coeffs = np.array([0.001, -0.12, +3]).reshape(-1,1)
continuation = X.T.dot(coeffs).T[0,:]
early = exercise[1,ITM] > continuation
payoff[1,ITM[early]] = exercise[1,ITM[early]]
payoff[2:,ITM[early]] = np.nan # Erase later exercises if any
print_paths(payoff, hide=np.nan);

,t=0dt,t=1dt,t=2dt,t=3dt,t=4dt
trajectory 1,,1.27,,,
trajectory 2,,,,3.11,
trajectory 3,,,1.44,,
trajectory 4,,2.22,,,
trajectory 5,,,,,0.0


Once you have this at hand, you can answer question a) by extracting `ex_dates` as follows:

In [16]:
ex_dates = [int(np.argwhere(~np.isnan(payoff[:,pn]))[0,0]) for pn in range(5)]
ex_dates

[1, 3, 2, 1, 4]

And question b) by extracting the payoff at these dates and discounting using the proper power of the discount rate:

In [17]:
payoff_t = payoff[ex_dates,range(5)]
call_price = (payoff_t*np.power(disc,ex_dates)).mean()
float(call_price)

1.600173223538422

### Alternative approach

Now, the answer above is correct, but somewhat painful to implement, having to erase "future" exercises at each point, keep track of the power for the discount rate, etc. It is not the most computationnaly efficient. Alternatively, consider the approach below.

In [18]:
question = np.array([
    [np.nan,np.nan,np.nan],
    [0.001, -0.12, +3],
    [0.001, -0.08, -1],
    [0.002, -0.12, -7]]) # For simplicity; would have to come from regression in a actual simu

# Initialize the value of the derivative to the final payoff
amer = exercise[-1,:]

As you can see, I will be working with a single vector. If you need only the price (question b), you do not even need keep track of these values as they are updated. To study the exercise frontier, for instance (or to answer question b), the `value` array below comes in handy. Otherwise, it can also serve for illustration purposes, when efficiency is not an issue.

In [19]:
tn = 4
print('At month %d:'%tn)
values = np.full(stock.shape,np.nan) 
values[tn,:] = amer
print_paths(values, hide=np.nan);

At month 4:


,t=0dt,t=1dt,t=2dt,t=3dt,t=4dt
trajectory 1,,,,,2.14
trajectory 2,,,,,2.90
trajectory 3,,,,,0.00
trajectory 4,,,,,4.05
trajectory 5,,,,,0.00


In [20]:
for tn in range(3,0,-1):
    print('At month %d:'%tn)
    ITM = np.where(exercise[tn,:] > 0)[0] # indices of the ITM trajectories
    S = stock[tn,ITM]
    
    # At this point, amer was not updated yet: it's the next period value
    amer = disc*amer # Discount it to present value
    
    # y is trivially the next period's discounted value
    X = np.array([S**2, S, np.ones(S.shape)])
    y = amer[ITM]
    #model = sm.OLS(y, X.T).fit() # In actual simulations
    #coeffs = model.params # In actual simulations
    coeffs = question[tn,:].reshape(-1,1) # Here take them from question
    
    # Override the discounted value of future payoff with the current 
    # exercise value when appropriate
    continuation = X.T.dot(coeffs).T[0,:]
    early = exercise[tn,ITM] > continuation    
    amer[ITM[early]] = exercise[tn,ITM[early]]
    
    # Just for illustration purposes
    values[tn,:] = amer
    print_paths(values, hide=np.nan);
    
# Finally, discount to time 0
amer = disc*amer
call_price = amer.mean()
print('b) Call price:',call_price)

At month 3:


,t=0dt,t=1dt,t=2dt,t=3dt,t=4dt
trajectory 1,,,,2.134657,2.14
trajectory 2,,,,3.110000,2.90
trajectory 3,,,,0.000000,0.00
trajectory 4,,,,1.550000,4.05
trajectory 5,,,,0.000000,0.00


At month 2:


,t=0dt,t=1dt,t=2dt,t=3dt,t=4dt
trajectory 1,,,2.129327,2.134657,2.14
trajectory 2,,,3.102235,3.110000,2.90
trajectory 3,,,1.440000,0.000000,0.00
trajectory 4,,,3.290000,1.550000,4.05
trajectory 5,,,0.000000,0.000000,0.00


At month 1:


,t=0dt,t=1dt,t=2dt,t=3dt,t=4dt
trajectory 1,,1.270000,2.129327,2.134657,2.14
trajectory 2,,3.094489,3.102235,3.110000,2.90
trajectory 3,,1.436404,1.440000,0.000000,0.00
trajectory 4,,2.220000,3.290000,1.550000,4.05
trajectory 5,,0.000000,0.000000,0.000000,0.00


b) Call price: 1.600173223538422


In [21]:
do_exercise = (exercise>0) & (exercise==values)
print_paths(do_exercise);

months = np.arange(do_exercise.shape[0])
first_date = lambda ex : months[ex][0] if ex.any() else len(ex)-1
print('a) Exercise dates:', [first_date(ex) for ex in do_exercise.T])

,t=0dt,t=1dt,t=2dt,t=3dt,t=4dt
trajectory 1,False,True,False,False,True
trajectory 2,False,False,False,True,True
trajectory 3,False,False,True,False,False
trajectory 4,False,True,True,True,True
trajectory 5,False,False,False,False,False


a) Exercise dates: [np.int64(1), np.int64(3), np.int64(2), np.int64(1), 4]


Note that when analyzing the exercise frontier implied by the method, the `values` matrix (which in turn yields `do_exercise`)may come handy...

In [22]:
ex_stock = np.where(do_exercise, stock, np.nan)
print_paths(ex_stock, hide=np.nan);

,t=0dt,t=1dt,t=2dt,t=3dt,t=4dt
trajectory 1,,101.27,,,102.14
trajectory 2,,,,103.11,102.9
trajectory 3,,,101.44,,
trajectory 4,,102.22,103.29,101.55,104.05
trajectory 5,,,,,


With a very large number of trajectories, the smallest (largest) value of the stock at which we exercised the call (put) on month $t$ could be see as a proxy of the exercise boundary at that point in time.